In [ ]:
import pandas as pd

In [ ]:
df = pd.read_excel('evaluation/inhibition_prediction/spectra/output/spectra_results.xlsx', sheet_name='Sheet1', header = None)

In [ ]:
df

In [ ]:
# Create a new DataFrame to restructure the data
new_data = []

# Loop through the DataFrame to gather relevant data for each model
for i in range(len(df)//9):  # Assuming each block is 8 rows long
    if i == 1:
        res = df.loc[9*i+1:9*(i+1)] 
    elif i == 2:
        res = df.loc[9*i+2:9*(i+2)] 
    else:
        res = df.loc[9*i:9*(i+1)]

    for offset in [0,7]:
        model_name = res.iloc[0,0+offset]
        all_sp = res.iloc[1,1+offset:5+offset].values
        all_mean = res.iloc[7,1+offset:5+offset].values
        all_std = res.iloc[8,1+offset:5+offset].values
    
        for model, sp, mean, std in zip([model_name]*len(all_sp), all_sp, all_mean, all_std):
            new_data.append(
                {
                    'Model_Name': model,
                    #This is cause it should be reversed per the paper, and it was in 20 bins
                    'Spectral_Parameter': 1-sp/20,
                    'Mean Performance': mean,
                    'STD Performance': std
                }
        
            )

# Create the new DataFrame
final_df = pd.DataFrame(new_data)
final_df

In [ ]:
import seaborn as sns
import numpy as np
from scipy import integrate
import matplotlib.pyplot as plt 
import matplotlib

colors = sns.color_palette("colorblind")
palette_dict = {model: color for model, color in zip(['Chemprop', 'EGNN', 'Transformer-M',
                                                    'FFNN', 'Random Forest'],colors)}
colors = palette_dict

legend = {f"Chemprop": "Chemprop", 
                 f"EGNN": "EGNN",
                 f"Transformer-M":"Transformer-M",
                 f"FFNN":"FFNN",
                 f"Random Forest":"Random Forest",
                 }
legend_placement = (1, 1)
x_axis_name = 'Spectral Parameter'
y_axis_name = 'AUC'

def calculate_aumc(y, x):
    return integrate.simps(y, x)

metric = ['Mean Performance','STD Performance']
fig, ax = plt.subplots(figsize=(10,10))

plt.rcParams['axes.labelsize'] = 20   # Axis labels
plt.rcParams['xtick.labelsize'] = 20  # X-axis tick labels
plt.rcParams['ytick.labelsize'] = 20  # Y-axis tick labels
plt.rcParams['legend.fontsize'] = 20

no_evi_df = final_df[final_df['Model_Name'] != 'Evidential Chemprop']

for key, group in no_evi_df.groupby('Model_Name'):
    #Problem with negative values since we have to sample at more consistent intervals
    result = -1*calculate_aumc(group[metric[0]], group['Spectral_Parameter'])
    first_value = -1*calculate_aumc(group[metric[0]]-group[metric[1]], group['Spectral_Parameter'])
    second_value = -1*calculate_aumc(group[metric[0]]+group[metric[1]], group['Spectral_Parameter'])
    
    AUSPC_range = f" (AUSPC: {result:0.2f}({first_value:0.2f}-{second_value:0.2f}))"
    
    if legend:
        group.plot('Spectral_Parameter', metric[0] , yerr = metric[1],
               label = legend[key]+AUSPC_range, ax=ax, marker='o', linestyle='', markersize = 5,
                  color = colors[legend[key]])
    else:
        group.plot('Spectral_Parameter', metric[0] , yerr = metric[1],
               label = key+AUSPC_range, ax=ax, marker='o', linestyle='', markersize = 5,
                  color = colors[legend[key]])
if legend_placement:   
    plt.legend(bbox_to_anchor=legend_placement)
if x_axis_name:
    plt.xlabel(x_axis_name)
if y_axis_name:
    plt.ylabel(y_axis_name)
# if title:
#     plt.title(title)

ax.spines[['right','top']].set_visible(False)
box = ax.get_position()
# ax.set_position([box.x0, box.y0, box.width * 0.5, box.height])
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
# ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.1), )
# plt.axhline(y=1.0, color='r', linestyle='--')
# plt.axhline(y=0.5, color='r', linestyle='--')
plt.savefig('evaluation/inhibition_prediction/spectra/output/SPECTRA_curve_no_evi.png', bbox_inches='tight')


In [ ]:
import seaborn as sns
import numpy as np
from scipy import integrate
import matplotlib.pyplot as plt 
import matplotlib

colors = sns.color_palette("colorblind")
palette_dict = {model: color for model, color in zip(['Chemprop', 'Chemprop (Evidential Learning)'],colors)}
colors = palette_dict

legend = {f"Chemprop": "Chemprop", 
                 f"Evidential Chemprop": "Chemprop (Evidential Learning)",
                 }
legend_placement = (1, 1)
x_axis_name = 'Spectral Parameter'
y_axis_name = 'AUC'

def calculate_aumc(y, x):
    return integrate.simps(y, x)

metric = ['Mean Performance','STD Performance']
fig, ax = plt.subplots(figsize=(10,10))

plt.rcParams['axes.labelsize'] = 20   # Axis labels
plt.rcParams['xtick.labelsize'] = 20  # X-axis tick labels
plt.rcParams['ytick.labelsize'] = 20  # Y-axis tick labels
plt.rcParams['legend.fontsize'] = 20

evi_df = final_df[final_df['Model_Name'].isin(['Chemprop','Evidential Chemprop'])]

for key, group in evi_df.groupby('Model_Name'):
    #Problem with negative values since we have to sample at more consistent intervals
    result = -1*calculate_aumc(group[metric[0]], group['Spectral_Parameter'])
    first_value = -1*calculate_aumc(group[metric[0]]-group[metric[1]], group['Spectral_Parameter'])
    second_value = -1*calculate_aumc(group[metric[0]]+group[metric[1]], group['Spectral_Parameter'])
    
    AUSPC_range = f" (AUSPC: {result:0.2f}({first_value:0.2f}-{second_value:0.2f}))"
    
    if legend:
        group.plot('Spectral_Parameter', metric[0] , yerr = metric[1],
               label = legend[key]+AUSPC_range, ax=ax, marker='o', linestyle='', markersize = 5,
                  color = colors[legend[key]])
    else:
        group.plot('Spectral_Parameter', metric[0] , yerr = metric[1],
               label = key+AUSPC_range, ax=ax, marker='o', linestyle='', markersize = 5,
                  color = colors[legend[key]])
if legend_placement:   
    plt.legend(bbox_to_anchor=legend_placement)
if x_axis_name:
    plt.xlabel(x_axis_name)
if y_axis_name:
    plt.ylabel(y_axis_name)
# if title:
#     plt.title(title)

ax.spines[['right','top']].set_visible(False)
box = ax.get_position()
# ax.set_position([box.x0, box.y0, box.width * 0.5, box.height])
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
for text in legend.get_texts():
    text.set_fontweight('bold')
    text.set_fontsize(20)
# ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.1), )
# plt.axhline(y=1.0, color='r', linestyle='--')
# plt.axhline(y=0.5, color='r', linestyle='--')
plt.savefig('evaluation/inhibition_prediction/spectra/output/SPECTRA_curve_evi.png', bbox_inches='tight')


In [ ]:
import seaborn as sns
import numpy as np
from scipy import integrate
import matplotlib.pyplot as plt 
import matplotlib

line_width = 5

colors = sns.color_palette("colorblind")
palette_dict = {model: color for model, color in zip(['DMPNN', 'Evidential DMPNN', 'EGNN', 'Transformer-M',
                                                    'FFNN', 'Random Forest'],colors)}
colors = palette_dict

legend = {f"Chemprop": "DMPNN", 
                 f"Evidential Chemprop": "Evidential DMPNN",
                 f"EGNN": "EGNN",
                 f"Transformer-M":"Transformer-M",
                 f"FFNN":"FFNN",
                 f"Random Forest":"Random Forest",
                 }
legend_placement = (1, 1)
x_axis_name = 'Spectral Parameter'
y_axis_name = 'AUC'

def calculate_aumc(y, x):
    return integrate.simps(y, x)

metric = ['Mean Performance','STD Performance']
fig, ax = plt.subplots(figsize=(10,10))
for spine in ax.spines.values():
    spine.set_linewidth(line_width)

plt.rcParams['axes.labelsize'] = 20   # Axis labels
plt.rcParams['xtick.labelsize'] = 20  # X-axis tick labels
plt.rcParams['ytick.labelsize'] = 20  # Y-axis tick labels
plt.rcParams['legend.fontsize'] = 20

model_order = ['Chemprop', 'EGNN', 'FFNN', 'Transformer-M', 'Random Forest', 'Evidential Chemprop']
sort_df = final_df.copy()

from pandas.api.types import CategoricalDtype

model_cat = CategoricalDtype(categories=model_order, ordered=True)
sort_df['Model_Name'] = sort_df['Model_Name'].astype(model_cat)
sort_df= sort_df.sort_values('Model_Name').reset_index(drop=True)

# print(sort_df)

for key, group in final_df.groupby('Model_Name'):
    #Problem with negative values since we have to sample at more consistent intervals
    print(group[metric[0]])
    print(group['Spectral_Parameter'])
    result = -1*calculate_aumc(group[metric[0]], group['Spectral_Parameter'])
    first_value = -1*calculate_aumc(group[metric[0]]-group[metric[1]], group['Spectral_Parameter'])
    second_value = -1*calculate_aumc(group[metric[0]]+group[metric[1]], group['Spectral_Parameter'])
    
    AUSPC_range = f" - {result:0.2f}[{first_value:0.2f}-{second_value:0.2f}]"
    
    if legend:
        group.plot('Spectral_Parameter', metric[0] , yerr = metric[1],
               label = legend[key]+AUSPC_range, ax=ax, marker='o', linestyle='', markersize = line_width*2, lw=line_width,
                  color = colors[legend[key]])
    else:
        group.plot('Spectral_Parameter', metric[0] , yerr = metric[1],
               label = key+AUSPC_range, ax=ax, marker='o', linestyle='', markersize = line_width*2, lw = line_width,
                  color = colors[legend[key]])
if legend_placement:   
    plt.legend(bbox_to_anchor=legend_placement)
if x_axis_name:
    plt.xlabel(x_axis_name)
if y_axis_name:
    plt.ylabel(y_axis_name)
# if title:
#     plt.title(title)

ax.spines[['right','top']].set_visible(False)
box = ax.get_position()
# ax.set_position([box.x0, box.y0, box.width * 0.5, box.height])
legend = ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), frameon=False)
for text in legend.get_texts():
    text.set_fontweight('bold')
    text.set_fontsize(20)
# ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.1), )
# plt.axhline(y=1.0, color='r', linestyle='--')
# plt.axhline(y=0.5, color='r', linestyle='--')
plt.savefig('evaluation/inhibition_prediction/spectra/output/SPECTRA_curve.png', bbox_inches='tight')
plt.savefig('evaluation/inhibition_prediction/spectra/output/SPECTRA_curve.pdf', bbox_inches='tight')



In [ ]:
import seaborn as sns
import numpy as np
from scipy import integrate
import matplotlib.pyplot as plt
import matplotlib
import pandas as pd
from pandas.api.types import CategoricalDtype

# Settings
line_width = 5
plt.rcParams['axes.labelsize'] = 20
plt.rcParams['xtick.labelsize'] = 20
plt.rcParams['ytick.labelsize'] = 20
plt.rcParams['legend.fontsize'] = 20

# Color setup
colors = sns.color_palette("colorblind")
model_order = ['Chemprop', 'EGNN', 'FFNN', 'Transformer-M', 'Random Forest', 'Evidential Chemprop']
display_names = ['DMPNN', 'EGNN', 'FFNN', 'Transformer-M', 'Random Forest', 'Evidential DMPNN']
palette_dict = {model: color for model, color in zip(display_names, colors)}
colors = palette_dict

# Display name mapping
legend = {
    'Chemprop': 'DMPNN',
    'Evidential Chemprop': 'Evidential DMPNN',
    'EGNN': 'EGNN',
    'Transformer-M': 'Transformer-M',
    'FFNN': 'FFNN',
    'Random Forest': 'Random Forest'
}

legend_placement = (1, 1)
x_axis_name = 'Spectral Parameter'
y_axis_name = 'AUC'
metric = ['Mean Performance', 'STD Performance']

# Prepare figure
fig, ax = plt.subplots(figsize=(10, 10))
for spine in ax.spines.values():
    spine.set_linewidth(line_width)

# Categorize and sort DataFrame
sort_df = final_df.copy()
model_cat = CategoricalDtype(categories=model_order, ordered=True)
sort_df['Model_Name'] = sort_df['Model_Name'].astype(model_cat)
sort_df = sort_df.sort_values('Model_Name').reset_index(drop=True)

# Group by in custom order
grouped = final_df.groupby('Model_Name')
for model in model_order:
    if model not in grouped.groups:
        continue
    group = grouped.get_group(model)

    # AUMC calculations
    result = -1 * integrate.simps(group[metric[0]], group['Spectral_Parameter'])
    first_value = -1 * integrate.simps(group[metric[0]] - group[metric[1]], group['Spectral_Parameter'])
    second_value = -1 * integrate.simps(group[metric[0]] + group[metric[1]], group['Spectral_Parameter'])

    AUSPC_range = f" - {result:0.2f}[{first_value:0.2f}-{second_value:0.2f}]"
    label = f"{legend[model]}{AUSPC_range}"
    color = colors[legend[model]]

    group.plot(
        x='Spectral_Parameter',
        y=metric[0],
        yerr=metric[1],
        label=label,
        ax=ax,
        marker='o',
        linestyle='',
        markersize=line_width * 2,
        lw=line_width,
        color=color
    )

# Axes and legend styling
ax.set_xlabel(x_axis_name, 
              fontsize=20, 
              #fontweight='bold'
              )
ax.set_ylabel(y_axis_name, 
              fontsize=20, 
              # fontweight='bold'
              )
ax.spines[['right', 'top']].set_visible(False)

# Legend
legend_obj = ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), frameon=False)
for text in legend_obj.get_texts():
    text.set_fontweight('bold')
    text.set_fontsize(20)

# Save
plt.savefig('evaluation/inhibition_prediction/spectra/output/SPECTRA_curve.png', bbox_inches='tight')
plt.savefig('evaluation/inhibition_prediction/spectra/output/SPECTRA_curve.pdf', bbox_inches='tight')


In [ ]:
final_df